In [1]:
import pandas as pd

# 1. Load the CSV files
# Ensure the files are in the current working directory of your Colab environment
df_abbr = pd.read_csv('/content/drive/MyDrive/连环画/Ref002/Abbreviations.csv')
df_pubs = pd.read_csv('/content/drive/MyDrive/连环画/Ref002/Ref002_Publishers.csv')

# 2. Create a dictionary mapping the abbreviation to the full publisher name
# Key: 本书简称 (Abbreviation)
# Value: 出版单位全称 (Full Name)
abbr_mapping = dict(zip(df_abbr['本书简称'], df_abbr['出版单位全称']))

# 3. Replace the entries in Ref002_Publishers.csv using the mapping dictionary
# The .replace() function will look for exact matches of the keys and swap them with the values
df_pubs['Publisher_1'] = df_pubs['Publisher_1'].replace(abbr_mapping)
df_pubs['Publisher_2'] = df_pubs['Publisher_2'].replace(abbr_mapping)

# 4. Save the updated dataframe to a new CSV file
output_filename = 'Ref002_Publishers_Updated.csv'
df_pubs.to_csv(output_filename, index=False, encoding='utf-8')

print(f"Replacement complete. The updated data has been saved to {output_filename}")

Replacement complete. The updated data has been saved to Ref002_Publishers_Updated.csv


In [2]:
import pandas as pd
import difflib

# 1. Load the original and updated CSV files
df_orig = pd.read_csv('/content/drive/MyDrive/连环画/Ref002/Ref002_Publishers.csv')
df_updated = pd.read_csv('/content/Ref002_Publishers_Updated.csv')

# 2. Define a function to compare two strings and wrap the differences in brackets
def bracket_differences(orig_text, updated_text):
    # Handle empty/NaN cells gracefully
    if pd.isna(orig_text) or pd.isna(updated_text):
        return updated_text

    orig_text = str(orig_text)
    updated_text = str(updated_text)

    # If they are exactly the same, return the updated text as-is
    if orig_text == updated_text:
        return updated_text

    # SequenceMatcher compares the two strings to find the matching blocks
    matcher = difflib.SequenceMatcher(None, orig_text, updated_text)
    result = ""

    # Iterate through the operations needed to turn orig_text into updated_text
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == 'equal':
            # Text is the same in both, add it without brackets
            result += updated_text[j1:j2]
        elif tag in ('insert', 'replace'):
            # Text is new or changed, wrap it in square brackets
            result += f"[{updated_text[j1:j2]}]"

    return result

# 3. Apply the comparison function to the relevant columns
# We use zip to pair up the original and updated cells row-by-row
for col in ['Publisher_1', 'Publisher_2']:
    df_updated[col] = [
        bracket_differences(orig, updated)
        for orig, updated in zip(df_orig[col], df_updated[col])
    ]

# 4. Save to a final output file
final_output = 'Ref002_Publishers_Final_Bracketed.csv'
df_updated.to_csv(final_output, index=False, encoding='utf-8-sig')

print(f"Comparison complete! Bracketed differences saved to {final_output}")

Comparison complete! Bracketed differences saved to Ref002_Publishers_Final_Bracketed.csv
